In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
import psycopg2
from scipy.stats import zscore
#from scattermap import scattermap

from scipy.stats import mannwhitneyu, zscore
from statsmodels.stats.multitest import multipletests
import numpy as np
from scipy.stats import spearmanr

In [ ]:
def compare_lists(list_a, list_b, option="common"):
    a, b = set(list_a), set(list_b)
    
    return list({
        "common": a & b,
        "uncommon": a ^ b,
        "diff_a": a - b,
        "diff_b": b - a
    }.get(option, a & b))

In [ ]:

data_table = pd.read_csv("data/endophenotype_prs_apoe_adjusted.csv")

print(data_table.shape)
data_table.head()

In [ ]:
pred_df = pd.read_csv('results/conclusion_macro_features.csv')
print(pred_df.shape)
pred_df.head()

In [ ]:
common_ids = compare_lists(data_table['donor_id'].to_list(),pred_df['donor_id'].to_list())
len(common_ids)

In [ ]:
prs_df = data_table[data_table['donor_id'].isin(common_ids)]
path_df_temp = pred_df[pred_df['donor_id'].isin(common_ids)]

print(prs_df.shape,path_df_temp.shape)

In [ ]:
#path_df = pd.merge()

path_df = path_df_temp.merge(
        diag_table[['donor_id','nd_current']],
        on='donor_id',
        how="left",
        #suffixes=("_df1", "_df2")
    )
#path_df.rename_
print(path_df.shape)
path_df.head()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from scripts.utils_fig import plot_correlation_scattermap, harmonize_plot_data, harmonize_plot_data_columns
from scripts.rename_var import fig_dict, prs_keys, bar_disease

def prepare_correlation_data(
    path_df,
    prs_df,
    path_cols,
    prs_cols,
    donor_col="donor_id",
    prs_label_map=None,
    min_n=6,
    min_dot_size=50,
    max_dot_size=300,
    sig_threshold=0.05,
    star_scale=0.7,
    n_perm=1000,
    random_state=42,
):
    # keep only existing columns
    path_cols = [c for c in path_cols if c in path_df.columns]
    prs_cols = [c for c in prs_cols if c in prs_df.columns]
    print(len(path_cols))

    merged = pd.merge(
        path_df[[donor_col] + path_cols],
        prs_df[[donor_col] + prs_cols],
        on=donor_col,
        how="inner"
    ).drop_duplicates(donor_col)
    
    # may remove , to ensure column should be numeric
    for col in path_cols + prs_cols:
        merged[col] = pd.to_numeric(merged[col], errors="coerce")

    corr_matrix = pd.DataFrame(index=path_cols, columns=prs_cols, dtype=float)
    pval_matrix = pd.DataFrame(index=path_cols, columns=prs_cols, dtype=float)

    rng = np.random.default_rng(random_state)

    # --------------------------------------------------
    # Correlation loop + permutation p-value
    # --------------------------------------------------
    for g_f in prs_cols:
        for p_f in path_cols:
            mask = merged[[p_f, g_f]].notna().all(axis=1).to_numpy()
            n_obs = int(mask.sum())

            if n_obs < min_n:
                corr_matrix.loc[p_f, g_f] = np.nan
                pval_matrix.loc[p_f, g_f] = 1.0
                continue

            x = merged.loc[mask, p_f].to_numpy()
            y = merged.loc[mask, g_f].to_numpy()

            # skip constant / no-variation variables
            if np.nanstd(x) == 0 or np.nanstd(y) == 0:
                corr_matrix.loc[p_f, g_f] = np.nan
                pval_matrix.loc[p_f, g_f] = 1.0
                continue

            obs_rho, _ = spearmanr(x, y)
            corr_matrix.loc[p_f, g_f] = obs_rho

            perm_rhos = np.empty(n_perm, dtype=float)

            for i in range(n_perm):
                y_perm = rng.permutation(y)
                perm_rhos[i], _ = spearmanr(x, y_perm)

            p = (np.sum(np.abs(perm_rhos) >= abs(obs_rho)) + 1) / (n_perm + 1)
            pval_matrix.loc[p_f, g_f] = p

    # --------------------------------------------------
    # FDR correction
    # --------------------------------------------------
    flat_p = pval_matrix.to_numpy().ravel()
    _, p_adj, _, _ = multipletests(flat_p, method="fdr_bh")

    padj_matrix = pd.DataFrame(
        p_adj.reshape(pval_matrix.shape),
        index=pval_matrix.index,
        columns=pval_matrix.columns
    )

    # --------------------------------------------------
    # Rename PRS columns
    # --------------------------------------------------
    if prs_label_map:
        new_cols = [prs_label_map.get(c, c) for c in prs_cols]

        corr_matrix.columns = new_cols
        pval_matrix.columns = new_cols
        padj_matrix.columns = new_cols
    else:
        new_cols = prs_cols

    # --------------------------------------------------
    # Dot size
    # --------------------------------------------------
    logp = -np.log10(padj_matrix.replace(0, 1e-12)).fillna(0)

    lo = logp.min().min()
    hi = logp.max().max()

    if hi == lo:
        dot_size = pd.DataFrame(
            min_dot_size,
            index=logp.index,
            columns=logp.columns
        )
    else:
        scaled = (logp - lo) / (hi - lo)
        dot_size = min_dot_size + scaled * (max_dot_size - min_dot_size)

    # --------------------------------------------------
    # Significance + star size
    # --------------------------------------------------
    sig_mask = (padj_matrix < sig_threshold).astype(int)
    star_size = sig_mask * dot_size * star_scale

    # --------------------------------------------------
    # Row counts
    # --------------------------------------------------
    sample_counts = [merged[c].notna().sum() for c in path_cols]

    return {
        "corr_matrix": corr_matrix,
        "pval_matrix": pval_matrix,
        "padj_matrix": padj_matrix,
        "dot_size": dot_size,
        "sig_mask": sig_mask,
        "star_size": star_size,
        "sample_counts": sample_counts,
        "p_min": float(np.nanmin(p_adj)),
        "p_max": float(np.nanmax(p_adj)),
        "path_labels": path_cols,
        "prs_labels": new_cols,
    }



In [ ]:
prs_grp = 'Diseases'
#prs_cols = list(prs_keys[prs_grp].keys())
path_cols = feature_cols.copy()
#prs_label_map = prs_keys[prs_grp]
prs_label_map = shade_mapping.copy()

plot_data = prepare_correlation_data(
    path_df=path_df,
    prs_df=prs_df,
    path_cols=path_cols,
    prs_cols=prs_cols,
    donor_col="donor_id",
    prs_label_map=prs_label_map,
    min_n=6,
    n_perm=10000
)
plot_data.keys()


In [ ]:
import pickle 

with open('results/saved_plot_data_perm_10K(shade_imputed_apoe_correct).pkl', 'wb') as f:
    pickle.dump(plot_data, f)